In [25]:
from pathlib import Path
import re
import pandas as pd

In [26]:
log_path = Path("../datasets/raw/unified_1m.log")

with log_path.open("r") as f:
    lines = f.readlines()

len(lines)

15323

In [27]:
header_pattern = re.compile(
    r'^(?P<timestamp>\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2}\.\d+(?:[+-]\d{4})?)\s+'
    r'(?P<thread>0x[0-9a-fA-F]+)\s+'
    r'(?P<type>\S+)\s+'
    r'(?P<activity>0x[0-9a-fA-F]+)\s+'
    r'(?P<pid>\d+)\s+'
    r'(?P<ttl>\d+)\s+'
    r'(?P<raw_message>.*)$'
)

In [28]:
events = []
current_event = None
current_message = []

for line in lines:
    match = header_pattern.match(line)

    if match:
        if current_event:
            current_event["raw_message"] = " ".join(current_message).strip()
            events.append(current_event)

        current_event = match.groupdict()
        current_message = [current_event["raw_message"]]

    else:
        if current_event:
            current_message.append(line.strip())

if current_event:
    current_event["raw_message"] = " ".join(current_message).strip()
    events.append(current_event)

In [29]:
len(events)
events[0]

{'timestamp': '2026-06-22 20:00:19.727566+0100',
 'thread': '0x17b458a',
 'type': 'Default',
 'activity': '0x0',
 'pid': '418',
 'ttl': '0',
 'raw_message': 'WindowServer: (HSTouchHIDService) [com.apple.Multitouch:Plugin] [TP] Dispatching digitizer event with 2 children, _eventMask=0x23 _childEventMask=0x22 Cancel=0 Touching=1 inRange=1'}

In [30]:
df = pd.DataFrame(events)
df.tail()

,timestamp,thread,type,activity,pid,ttl,raw_message
12794,2026-06-22 20:01:19.394319+0100,0x17b5ba6,Activity,0x226ac0c,392,0,locationd: CL: _CLDaemonGetAppsUsingLocation
12795,2026-06-22 20:01:19.466872+0100,0x17b60bb,Activity,0x226ac82,88596,0,iTerm2: (CoreSpotlight) end-index-batch
12796,2026-06-22 20:01:19.468472+0100,0x17b60bb,Activity,0x226ac83,88596,0,iTerm2: (CoreSpotlight) index-items
12797,2026-06-22 20:01:19.676755+0100,0x17b0d82,Error,0x0,0,0,kernel: (Sandbox) Sandbox: logd_helper(6702) d...
12798,2026-06-22 20:01:19.677093+0100,0x17b0d82,Error,0x0,0,0,kernel: (Sandbox) Sandbox: logd_helper(6702) d...


In [31]:
import re

paren_re = re.compile(r'\((.*?)\)')
bracket_re = re.compile(r'\[(.*?)\]')

def parse_message(msg: str):
    # 1. process (before first space or :)
    process = None
    rest = msg

    if ':' in msg:
        process, rest = msg.split(':', 1)
        process = process.strip()

    # 2. parentheses subsystem
    paren = paren_re.findall(rest)
    rest = paren_re.sub('', rest).strip()

    # 3. bracket tags
    tags = bracket_re.findall(rest)
    rest = bracket_re.sub('', rest).strip()

    return {
        "process": process,
        "subsystem": paren,
        "tags": tags,
        "message": rest.strip()
    }

In [32]:
df["parsed"] = df["raw_message"].apply(parse_message)

df = pd.concat(
    [df.drop(columns=["parsed"]), df["parsed"].apply(pd.Series)],
    axis=1
)

df.tail()

,timestamp,thread,type,activity,pid,ttl,raw_message,process,subsystem,tags,message
12794,2026-06-22 20:01:19.394319+0100,0x17b5ba6,Activity,0x226ac0c,392,0,locationd: CL: _CLDaemonGetAppsUsingLocation,locationd,[],[],CL: _CLDaemonGetAppsUsingLocation
12795,2026-06-22 20:01:19.466872+0100,0x17b60bb,Activity,0x226ac82,88596,0,iTerm2: (CoreSpotlight) end-index-batch,iTerm2,[CoreSpotlight],[],end-index-batch
12796,2026-06-22 20:01:19.468472+0100,0x17b60bb,Activity,0x226ac83,88596,0,iTerm2: (CoreSpotlight) index-items,iTerm2,[CoreSpotlight],[],index-items
12797,2026-06-22 20:01:19.676755+0100,0x17b0d82,Error,0x0,0,0,kernel: (Sandbox) Sandbox: logd_helper(6702) d...,kernel,"[Sandbox, 6702, 1]",[],Sandbox: logd_helper deny file-read-data /Appl...
12798,2026-06-22 20:01:19.677093+0100,0x17b0d82,Error,0x0,0,0,kernel: (Sandbox) Sandbox: logd_helper(6702) d...,kernel,"[Sandbox, 6702, 1]",[],Sandbox: logd_helper deny file-read-data /User...


In [33]:
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

In [34]:
df.head()

,timestamp,thread,type,activity,pid,ttl,raw_message,process,subsystem,tags,message
0,2026-06-22 19:00:19.727566+00:00,0x17b458a,Default,0x0,418,0,WindowServer: (HSTouchHIDService) [com.apple.M...,WindowServer,[HSTouchHIDService],"[com.apple.Multitouch:Plugin, TP]","Dispatching digitizer event with 2 children, _..."
1,2026-06-22 19:00:19.862505+00:00,0x17b5b03,Default,0x0,357,8,powerd: [com.apple.powerd:displayState] Deskto...,powerd,[],[com.apple.powerd:displayState],DesktopMode check on Battery 0
2,2026-06-22 19:00:19.862875+00:00,0x17b5b03,Default,0x0,357,8,powerd: [com.apple.powerd:displayState] Deskto...,powerd,[],[com.apple.powerd:displayState],DesktopMode check on Battery 0
3,2026-06-22 19:00:20.006890+00:00,0x17b0b92,Default,0x0,0,0,kernel: (com.apple.DriverKit-AppleBCMWLAN.dext...,kernel,"[com.apple.DriverKit-AppleBCMWLAN.dext, ]",[],IO80211PeerMonitor::checkForDPS Enter
4,2026-06-22 19:00:20.007193+00:00,0x17b0b92,Default,0x0,0,0,kernel: (com.apple.DriverKit-AppleBCMWLAN.dext...,kernel,[com.apple.DriverKit-AppleBCMWLAN.dext],[],LQM-WiFi: BE : Count: 45 avgLatencyMs:2 maxLat...


Normalize unified log timestamps to UTC so downstream statistics and ML notebooks can compare events on a consistent timeline.

In [ ]:
output_path = Path("../datasets/processed/unified_logs_processed.csv")
df.to_csv(output_path, index=False)